# vLLM + Redis Integration — Hands-On

**TechBot scenario:** Run inference locally with vLLM, with Redis as a semantic cache in front of it.

**Prerequisites:**
```bash
# Redis Stack
docker run -d --name redis-stack -p 6379:6379 -p 8001:8001 redis/redis-stack:latest

# vLLM (requires NVIDIA GPU + CUDA)
pip install vllm
python -m vllm.entrypoints.openai.api_server \
  --model meta-llama/Llama-3-8B-Instruct \
  --port 8000 \
  --enable-prefix-caching

# OR use a smaller model for testing:
# python -m vllm.entrypoints.openai.api_server \
#   --model facebook/opt-125m \
#   --port 8000
```

This notebook includes a **mock vLLM** so you can run everything without a GPU.

## Part 1: Talk to vLLM Directly

In [ ]:
import httpx
import time
import os

VLLM_URL = os.getenv("VLLM_URL", "http://localhost:8000")
VLLM_MODEL = os.getenv("VLLM_MODEL", "meta-llama/Llama-3-8B-Instruct")

def check_vllm_health() -> bool:
    try:
        resp = httpx.get(f"{VLLM_URL}/health", timeout=2.0)
        return resp.status_code == 200
    except Exception:
        return False

vllm_available = check_vllm_health()
print(f"vLLM available: {vllm_available}")
if not vllm_available:
    print("vLLM not running — will use mock LLM for this notebook.")
    print("Start vLLM if you have a GPU for real inference.")

In [ ]:
# Mock vLLM — simulates inference with realistic latency
import asyncio

MOCK_RESPONSES = {
    "password": "To reset your password, go to Settings > Security > Reset Password. You will receive an email with a reset link.",
    "install": "To install the SDK, run: pip install techbot-sdk. This works on Windows, Mac, and Linux.",
    "auth": "Set your API key as an environment variable: export TECHBOT_API_KEY=your_key. Then use Client() in Python.",
    "price": "TechBot Pro costs $29/month and includes 100,000 API calls. The free tier includes 1,000 calls/month.",
    "default": "Thank you for your question. Please contact support@techcorp.com for more help."
}

def mock_vllm(prompt: str, latency_ms: int = 800) -> str:
    """Simulates vLLM inference with a fixed latency."""
    time.sleep(latency_ms / 1000)
    prompt_lower = prompt.lower()
    for keyword, response in MOCK_RESPONSES.items():
        if keyword in prompt_lower:
            return response
    return MOCK_RESPONSES["default"]


def call_llm(prompt: str) -> tuple[str, int]:
    """Call real vLLM or fall back to mock."""
    if vllm_available:
        start = time.perf_counter()
        resp = httpx.post(
            f"{VLLM_URL}/v1/completions",
            json={"model": VLLM_MODEL, "prompt": prompt, "max_tokens": 256},
            timeout=120.0
        )
        latency = int((time.perf_counter() - start) * 1000)
        return resp.json()["choices"][0]["text"].strip(), latency
    else:
        start = time.perf_counter()
        response = mock_vllm(prompt)
        latency = int((time.perf_counter() - start) * 1000)
        return response, latency


# Test direct LLM call
response, latency = call_llm("How do I reset my password?")
print(f"Direct LLM call:")
print(f"  Latency: {latency}ms")
print(f"  Response: {response[:100]}")

## Part 2: Redis Semantic Cache in Front of vLLM

In [ ]:
from redisvl.extensions.llmcache import SemanticCache

REDIS_URL = "redis://localhost:6379"

cache = SemanticCache(
    name="vllm-gateway-cache",
    redis_url=REDIS_URL,
    distance_threshold=0.15,
    ttl=3600,
)

stats = {"hits": 0, "misses": 0, "total_saved_ms": 0}

def cached_inference(prompt: str) -> dict:
    """Redis semantic cache wrapper around the LLM."""
    start = time.perf_counter()
    
    # Check cache first
    results = cache.check(prompt=prompt)
    if results:
        elapsed = int((time.perf_counter() - start) * 1000)
        stats["hits"] += 1
        stats["total_saved_ms"] += 800 - elapsed  # assume 800ms LLM baseline
        return {
            "source": "cache",
            "response": results[0]["response"],
            "latency_ms": elapsed,
            "distance": float(results[0].get("vector_distance", 0))
        }
    
    # Cache miss — call LLM
    response, llm_latency = call_llm(prompt)
    cache.store(prompt=prompt, response=response)
    elapsed = int((time.perf_counter() - start) * 1000)
    stats["misses"] += 1
    
    return {
        "source": "vllm" if vllm_available else "mock",
        "response": response,
        "latency_ms": elapsed,
        "distance": None
    }


print("Running cached inference demo:")
print("=" * 60)

test_queries = [
    "How do I reset my password?",
    "I forgot my password, what do I do?",           # semantic hit
    "Can I recover my account after forgetting password?",  # semantic hit
    "How do I install the SDK?",
    "What is the installation command for the SDK?",  # semantic hit
    "How much does TechBot cost?",
    "What are the pricing plans?",                    # semantic hit
    "How do I authenticate with the API?",
]

for query in test_queries:
    result = cached_inference(query)
    source_icon = "⚡" if result["source"] == "cache" else "🤖"
    dist_str = f" (dist={result['distance']:.4f})" if result["distance"] is not None else ""
    print(f"{source_icon} [{result['source']:<6}] {result['latency_ms']:>5}ms{dist_str}")
    print(f"   Q: {query}")
    print(f"   A: {result['response'][:80]}..." if len(result['response']) > 80 else f"   A: {result['response']}")
    print()

In [ ]:
# Print cache performance summary
total = stats['hits'] + stats['misses']
hit_rate = stats['hits'] / total if total > 0 else 0

print(f"Cache Performance:")
print(f"  Total queries:  {total}")
print(f"  Cache hits:     {stats['hits']} ({hit_rate:.0%})")
print(f"  Cache misses:   {stats['misses']}")
print(f"  Time saved:     ~{stats['total_saved_ms']}ms total")
if total > 0:
    print(f"  Cost reduction: {hit_rate:.0%} fewer LLM calls")

## Part 3: vLLM Prefix Caching — Multi-Turn Benefit

This demonstrates WHY you should put shared content (system prompt, RAG context) BEFORE the user query in the prompt — so vLLM can reuse its KV cache across turns.

In [ ]:
# Prompt structure that maximizes vLLM prefix caching

SYSTEM_PROMPT = """You are TechBot, a helpful technical support assistant for TechCorp software.
Answer accurately using the documentation provided. If the answer is not in the docs, say so."""

RAG_CONTEXT = """Relevant documentation:
- To install: pip install techbot-sdk (Python 3.9+ required)
- Authentication: set TECHBOT_API_KEY environment variable
- Password reset: Settings > Security > Reset Password
- Pricing: Free (1,000 calls/mo), Pro $29/mo (100,000 calls)"""

def build_prompt(conversation_history: list, new_message: str) -> str:
    """
    Build prompt with shared prefix first.
    vLLM can reuse KV cache for the SYSTEM_PROMPT and RAG_CONTEXT sections
    across all requests that share them — only the history+message differs.
    """
    history_text = ""
    if conversation_history:
        history_text = "\n".join(
            f"{'User' if m['role'] == 'user' else 'Assistant'}: {m['content']}"
            for m in conversation_history
        )
    
    # ORDER MATTERS for prefix caching:
    # 1. System (shared across ALL users) → most cacheable
    # 2. RAG context (shared across SIMILAR queries) → very cacheable
    # 3. History (shared across turns of SAME session) → somewhat cacheable
    # 4. New message (unique to this request) → not cacheable
    
    return (
        f"System: {SYSTEM_PROMPT}\n"
        f"\n{RAG_CONTEXT}\n"
        + (f"\nConversation:\n{history_text}\n" if history_text else "")
        + f"\nUser: {new_message}\nAssistant:"
    )


# Multi-turn conversation
print("Multi-turn conversation (prefix caching benefits):")
print("=" * 60)

conversation = []

turns = [
    "How do I install the SDK?",
    "What about on Windows?",
    "And how do I authenticate?",
]

for turn_num, user_msg in enumerate(turns):
    prompt = build_prompt(conversation, user_msg)
    
    # Show prompt prefix length (shared prefix = cacheable by vLLM)
    system_section_len = len(f"System: {SYSTEM_PROMPT}\n\n{RAG_CONTEXT}\n")
    total_len = len(prompt)
    new_content_len = total_len - system_section_len
    
    print(f"Turn {turn_num + 1}: '{user_msg}'")
    print(f"  Prompt: {total_len} chars total")
    print(f"  Shared prefix: {system_section_len} chars (vLLM can cache this)")
    print(f"  New content:   {new_content_len} chars (must compute forward pass)")
    print(f"  Cache efficiency: {system_section_len/total_len:.0%} of tokens reusable")
    
    response, latency = call_llm(prompt)
    print(f"  Response ({latency}ms): {response[:60]}..." if len(response) > 60 else f"  Response ({latency}ms): {response}")
    
    conversation.append({"role": "user", "content": user_msg})
    conversation.append({"role": "assistant", "content": response})
    print()

## Part 4: FastAPI Gateway with Redis Cache + vLLM

The complete gateway pattern used in `05_chatbot_project/tasks/inference.py`:

In [ ]:
gateway_code = '''
# gateway.py — Redis semantic cache in front of vLLM
# This is what tasks/inference.py does inside the Celery worker

from fastapi import FastAPI
from redisvl.extensions.llmcache import SemanticCache
import httpx

app = FastAPI()

cache = SemanticCache(
    name="vllm-cache",
    redis_url="redis://localhost:6379",
    distance_threshold=0.15,
    ttl=3600,
)

@app.post("/v1/completions")
async def cached_completion(prompt: str, max_tokens: int = 512):
    """
    Drop-in proxy for vLLM with semantic caching.
    Point your client at this instead of vLLM directly.
    """

    # Step 1: Check semantic cache
    results = cache.check(prompt=prompt)
    if results:
        return {
            "source": "cache",
            "choices": [{"text": results[0]["response"]}]
        }

    # Step 2: Cache miss → forward to vLLM
    async with httpx.AsyncClient() as client:
        resp = await client.post(
            "http://localhost:8000/v1/completions",
            json={"model": "meta-llama/Llama-3-8B-Instruct",
                  "prompt": prompt, "max_tokens": max_tokens},
            timeout=120.0,
        )

    response_text = resp.json()["choices"][0]["text"]

    # Step 3: Store in cache for future similar queries
    cache.store(prompt=prompt, response=response_text)

    return {
        "source": "vllm",
        "choices": [{"text": response_text}]
    }

# Run: uvicorn gateway:app --port 9001
# Then point your client at port 9001 instead of 8000
'''
print(gateway_code)

## Cleanup

In [ ]:
import redis as redis_lib
r = redis_lib.Redis(host="localhost", port=6379, decode_responses=True)

cache.clear()
print("Cache cleared.")

remaining = list(r.scan_iter("vllm-gateway-cache*"))
print(f"Remaining keys: {len(remaining)}")

## Summary

### Two complementary caching layers:

| Layer | Where | What | Scope |
|-------|-------|------|-------|
| Redis SemanticCache | Application code | Complete LLM responses | Cross-user, persistent |
| vLLM Prefix Cache | GPU memory | KV attention tensors | Single node, in-memory |

### Prompt structure for maximum prefix caching benefit:

```
[System Prompt]           ← same for every user → 100% shared
[RAG Context]             ← same for similar questions → highly shared
[Conversation History]    ← same within a session → partially shared
[User Message]            ← unique per request → never cached
```

### TechBot complete stack:
1. Redis semantic cache: 0–5ms for cache hits, 0 LLM cost
2. Redis vector store: 1–3ms RAG retrieval, grounds answers in facts
3. Celery + Redis broker: non-blocking async, scales to N workers
4. vLLM prefix cache: 50–80% faster TTFT for multi-turn chats
5. Redis chat history: persistent across all workers, TTL-managed